In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Clone repo on first run, git pull on restarts (idempotent)
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo_dir = '/content/ebpf-fuzzing-thesis'

if not os.path.exists(repo_dir):
    ret = os.system(f'git clone https://{token}@github.com/Strhata/ebpf-fuzzing-thesis.git {repo_dir}')
    assert ret == 0, 'git clone failed'
else:
    ret = os.system(f'git -C {repo_dir} pull')
    assert ret == 0, 'git pull failed'

os.chdir(repo_dir)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — Install training-side dependencies (~10 min on a fresh runtime)
import subprocess
subprocess.run(
    ['pip', 'install', '-q', '-r', 'ml/requirements_colab.txt'],
    check=True,
)

In [ ]:
# Cell 4 — Config (the only cell you may need to edit)
G        = 8       # completions per prompt; fall back to 4 if smoke test OOMs
MAX_LEN  = 1200    # max completion tokens
RUN_NAME = 'grpo-depth-reward-v1'
OUTPUT_DIR  = f'/content/drive/MyDrive/{RUN_NAME}'
REWARD_URL  = userdata.get('REWARD_SERVER_URL')

print(f'G={G}  MAX_LEN={MAX_LEN}  RUN_NAME={RUN_NAME}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print(f'REWARD_URL={REWARD_URL}')

In [ ]:
# Cell 5 — Launch training (safe to re-run: --resume handles fresh start and continuation)
import os
from google.colab import userdata

os.environ['WANDB_API_KEY']  = userdata.get('WANDB_API_KEY')
os.environ['REWARD_API_KEY'] = userdata.get('REWARD_API_KEY')

os.system(
    f'python ml/rl_grpo.py'
    f' --resume'
    f' --remote-reward-url {REWARD_URL}'
    f' --run-name {RUN_NAME}'
    f' --num-generations {G}'
    f' --max-completion-length {MAX_LEN}'
    f' --output-dir {OUTPUT_DIR}'
)